In [1]:
import os

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

In [2]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer

d:\miniconda3\envs\PyTorch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
ds = Dataset.load_from_disk("../data/alpaca_data_zh/")
ds

Dataset({
    features: ['output', 'input', 'instruction'],
    num_rows: 26858
})

In [4]:
ds = ds.train_test_split(test_size=0.2)
ds

DatasetDict({
    train: Dataset({
        features: ['output', 'input', 'instruction'],
        num_rows: 21486
    })
    test: Dataset({
        features: ['output', 'input', 'instruction'],
        num_rows: 5372
    })
})

In [5]:
ds['train'][:3]

{'output': ['改进的代码段如下:\n\n```python\narray = [1, 2, 3, 4]\narray = [x+5 for x in array]\n```\n这里我们使用了列表解析（list comprehension）这种简洁的写法，可以更清晰地表达列表的创建，而且性能也更好。',
  '“机器的进化：我们的未来都交给人工智能了吗？”',
  '这个句子中使用了句号（。）。'],
 'input': ['输入：\narray = [1, 2, 3, 4]\nfor x in range(len(array)):\n  array[x] += 5',
  '',
  ''],
 'instruction': ['建议改进以下代码段，使其更符合Python风格。',
  '为这篇关于人工智能的文章想一个有趣的标题。',
  '识别以下句子中使用的标点符号类型：天气很热。']}

In [6]:
tokenizer = AutoTokenizer.from_pretrained("Langboat/bloom-1b4-zh")
tokenizer

BloomTokenizerFast(name_or_path='Langboat/bloom-1b4-zh', vocab_size=46145, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [7]:
def process_func(example):
    MAX_LENGTH = 256
    input_ids, attention_mask, labels = [], [], []
    instruction = tokenizer("\n".join(["Human: " + example["instruction"], example["input"]]).strip() + "\n\nAssistant: ")
    response = tokenizer(example["output"] + tokenizer.eos_token)
    input_ids = instruction["input_ids"] + response["input_ids"]
    attention_mask = instruction["attention_mask"] + response["attention_mask"]
    labels = [-100] * len(instruction["input_ids"]) + response["input_ids"]
    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [8]:
tokenized_ds = ds.map(process_func, remove_columns=ds['train'].column_names)
tokenized_ds

Map: 100%|██████████| 5372/5372 [00:01<00:00, 4839.12 examples/s]


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 21486
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 5372
    })
})

In [9]:
tokenizer.decode(tokenized_ds['train'][1]["input_ids"])

'Human: 为这篇关于人工智能的文章想一个有趣的标题。\n\nAssistant: “机器的进化：我们的未来都交给人工智能了吗？”</s>'

In [10]:
tokenizer.decode(list(filter(lambda x: x != -100, tokenized_ds['train'][1]["labels"])))

'“机器的进化：我们的未来都交给人工智能了吗？”</s>'

In [11]:
model = AutoModelForCausalLM.from_pretrained("Langboat/bloom-1b4-zh")

In [12]:
from peft import PromptEncoderConfig, TaskType, get_peft_model, PromptEncoderReparameterizationType

config = PromptEncoderConfig(task_type=TaskType.CAUSAL_LM, num_virtual_tokens=10,
                             encoder_reparameterization_type=PromptEncoderReparameterizationType.MLP,
                             encoder_dropout=0.1, encoder_num_layers=5, encoder_hidden_size=1024)
config

PromptEncoderConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.P_TUNING: 'P_TUNING'>, auto_mapping=None, peft_version='0.18.0', base_model_name_or_path=None, revision=None, inference_mode=False, num_virtual_tokens=10, token_dim=None, num_transformer_submodules=None, num_attention_heads=None, num_layers=None, modules_to_save=None, encoder_reparameterization_type=<PromptEncoderReparameterizationType.MLP: 'MLP'>, encoder_hidden_size=1024, encoder_num_layers=5, encoder_dropout=0.1)

In [13]:
model = get_peft_model(model, config)

d:\miniconda3\envs\PyTorch\Lib\site-packages\peft\tuners\p_tuning\model.py:105: UserWarning: for MLP, the argument `encoder_num_layers` is ignored. Exactly 2 MLP layers are used.
  warnings.warn(


In [14]:
model.print_trainable_parameters()

trainable params: 5,267,456 || all params: 1,308,379,136 || trainable%: 0.4026


In [15]:
args = TrainingArguments(
    output_dir="./chatbot",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    eval_strategy='steps',
    logging_steps=50,
    num_train_epochs=1
)

In [16]:
trainer = Trainer(
    model=model,
    args=args,
    tokenizer=tokenizer,
    train_dataset=tokenized_ds['train'].select(range(10000)),
    eval_dataset=tokenized_ds['test'].select(range(1000)),
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
)

C:\Users\10433\AppData\Local\Temp\ipykernel_17816\4211728216.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [17]:
trainer.train()

Step,Training Loss,Validation Loss
50,2.667500,2.551211
100,2.546100,2.496308
150,2.390000,2.468950
200,2.409500,2.455254
250,2.360600,2.449072
300,2.418500,2.432396
350,2.333600,2.425066
400,2.380100,2.413367
450,2.385200,2.407225
500,2.428600,2.398644


TrainOutput(global_step=1250, training_loss=2.3779669555664062, metrics={'train_runtime': 2000.3211, 'train_samples_per_second': 4.999, 'train_steps_per_second': 0.625, 'total_flos': 5508607046123520.0, 'train_loss': 2.3779669555664062, 'epoch': 1.0})

In [18]:
model = model.cuda()
ipt = tokenizer("Human: {}\n{}".format("数学考试有哪些技巧？", "").strip() + "\n\nAssistant: ", return_tensors="pt").to(model.device)
print(tokenizer.decode(model.generate(**ipt, max_length=256, do_sample=True)[0], skip_special_tokens=True))

Human: 数学考试有哪些技巧？

Assistant: 数学考试是一个较为严谨的领域，需要学生在学习与练习过程中注重多维度的考虑因素，并且以严谨、科学的方法进行计算，以此达到较高的数学成绩。在这里，我们可以提出一些建议以供参考。首先，在复习与练习数学问题上，我们建议使用适当的策略和技巧。例如，可以在练习和复习过程中利用图表、插图和视频等辅助工具，来帮助记忆和理解知识要点。此外，我们应该养成良好的学习习惯，比如，按时完成作业，对错误和疑惑进行整理，以及积极与同学交流讨论，并和老师取得联系。

最后，我们建议学生在考试之前做好足够多的模拟题练习，以找出自己的薄弱领域，并且制定相应的措施加以弥补。
